# Step 4. 후보 변수 생성

- 변수 정의: `src/features.py` (계산식과 필요 과거 기간은 `SPEC`에 기록)
- 규칙: t일까지의 정보만 사용, 롤링 윈도우 최대 60거래일, 가격은 비율로만 사용, 수급은 t일 장 마감 후 공표분을 lag 없이 사용
- **v2**: 가격은 KRX 정규장 일봉(`krx_daily_candles`)을 쓰고, 비정상 꼬리 보정은 하지 않는다(`clean=False`). 거래대금은 KRX 실제 거래대금이다.
- 수급 변수의 분모: 투자자별·기관 세부 매매와 신용·대차 일수는 통합 거래량(기존 일봉, KRX+NXT)으로, 공매도·프로그램 매매는 KRX 거래량으로 나눈다(`03b_krx_validation.ipynb` 4절).
- v2에서 추가된 변수: `log_mcap`(시가총액), `turnover_20`, `turnover_ratio_5_60`(회전율). KRX 일봉의 날짜별 상장주식수를 쓴다.
- 범위: 2024-01-02 ~ 2026-06-30 데이터로 계산하고, 분석 표본(2024-04-01~)의 구성종목 행에 붙인다.
- 산출물: `data/cache/features_train.parquet`, `outputs/feature_candidates.csv`(IC 등은 Step 5에서 채움)

In [1]:
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.data import load_table
from src import features as F

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 300)

S, E = "2024-01-01", "2026-06-30"
FLOW_TABLES = ["stock_investor_trading", "stock_institution_breakdown", "stock_program_trades", "stock_short_selling",
               "stock_credit_trades", "stock_securities_lending", "market_indicator_investor_trading"]

def load_inputs(end):
    c = load_table("krx_daily_candles", S, E)
    vol_all = load_table("stock_daily_candles", S, E)[["symbol", "trade_date", "volume"]]   # KRX+NXT 통합 거래량
    idx = load_table("market_indicator_daily_candles", S, E)
    flows = {t: load_table(t, S, E) for t in FLOW_TABLES}
    # end 이후 행을 잘라낸다(누수 검사용)
    cut = lambda d: d[d.trade_date <= end]
    return cut(c), cut(idx), {k: cut(v) for k, v in flows.items()}, cut(vol_all)

uni = pd.read_parquet(ROOT / "data" / "cache" / "universe_train.parquet")
industry = pd.read_parquet(ROOT / "data" / "cache" / "stocks_industry.parquet").set_index("symbol")["industry"]
c, idx, flows, vol_all = load_inputs(E)
feat, stock_cols, market_cols = F.build_all(c, idx, uni, flows, industry, clean=False, vol_all=vol_all)
print(f"변수: 종목 단위 {len(stock_cols)}개(+ 순위 버전 {len([k for k in feat.columns if k.endswith('_rk')])}개), 시장 공통 {len(market_cols)}개")
print(f"행: {len(feat):,} (유니버스 전체, 워밍업 포함)")

변수: 종목 단위 102개(+ 순위 버전 98개), 시장 공통 31개
행: 121,261 (유니버스 전체, 워밍업 포함)


## 1. 변수 명세 (계열별 개수와 필요 과거 기간)

In [2]:
spec = pd.DataFrame([(k, *v) for k, v in F.SPEC.items()], columns=["변수명", "계열", "계산식", "필요 과거 기간", "비고"])
spec["단위"] = np.where(spec.변수명.isin(market_cols), "날짜 공통", "종목")
assert spec["필요 과거 기간"].max() <= 61, "61거래일 초과 변수가 있다"
display(spec.groupby(["계열", "단위"]).agg(개수=("변수명", "size"), 최대_과거기간=("필요 과거 기간", "max")))
print("필요 과거 기간 최댓값:", spec["필요 과거 기간"].max(), "거래일")

,,개수,최대_과거기간
계열,단위,,
가격 위치,종목,5,59
거래·유동성,종목,10,59
공매도·대차,종목,6,20
변동성,종목,10,60
수급,종목,24,20
수익률·모멘텀,종목,8,60
시장,날짜 공통,22,60
시장 대비,종목,5,60
시장 수급,날짜 공통,9,19


필요 과거 기간 최댓값: 60 거래일


## 2. 누수 검사: 데이터 절단 테스트

2025-06-30 이후 데이터를 모두 지우고 다시 계산했을 때, 2025-06-30 이하 날짜의 변수 값이 전체 데이터로 계산한 값과 **정확히 같아야** 한다. 다르면 미래 값이 들어간 것이다.

In [3]:
CUT = "2025-06-30"
c2, idx2, flows2, vol_all2 = load_inputs(CUT)
feat2, _, _ = F.build_all(c2, idx2, uni[uni.trade_date <= CUT], flows2, industry, clean=False, vol_all=vol_all2)
a = feat[feat.trade_date <= CUT].set_index(["symbol", "trade_date"]).sort_index()
b = feat2.set_index(["symbol", "trade_date"]).sort_index()
cols = [k for k in a.columns if k in b.columns and a[k].dtype != object]
diff = {}
for k in cols:
    x, y = a[k].astype(float), b[k].astype(float)
    bad = ~((x - y).abs().le(1e-9) | (x.isna() & y.isna()))
    if bad.any():
        diff[k] = int(bad.sum())
print(f"비교 변수 {len(cols)}개, 비교 행 {len(a):,}")
print("값이 달라진 변수:", diff if diff else "없음 (누수 없음)")

비교 변수 232개, 비교 행 72,461
값이 달라진 변수: 없음 (누수 없음)


## 3. 결측률 (분석 표본)

In [4]:
fs = feat[feat.trade_date >= "2024-04-01"]
na = fs[stock_cols + market_cols].isna().mean()
print(f"결측이 없는 변수 {int((na == 0).sum())}개 / {len(na)}")
na[na > 0].sort_values(ascending=False).round(4).to_frame("결측률").head(30)

결측이 없는 변수 38개 / 133


,결측률
cfd_buy_rate,0.1675
fininv_net_1,0.0091
inst_net_1,0.0091
pension_net_1,0.0091
indiv_net_1,0.0091
foreign_net_1,0.0091
pension_net_5,0.0088
foreign_net_5,0.0088
inst_net_5,0.0088
fininv_net_5,0.0088


In [5]:
# 결측 원인 요약: 신규 상장(과거 일봉 부족), CFD 미제공, 거래정지
nh = fs.merge(uni[["symbol", "trade_date", "enough_hist_60"]], on=["symbol", "trade_date"])
print("과거 일봉 61개 미만 행에서 ret_60 결측률:", round(nh.loc[~nh.enough_hist_60, "ret_60"].isna().mean(), 3),
      "| 그 외:", round(nh.loc[nh.enough_hist_60, "ret_60"].isna().mean(), 4))

과거 일봉 61개 미만 행에서 ret_60 결측률: 1.0 | 그 외: 0.0


## 4. 분포 요약 (분석 표본, 분위수)

In [6]:
q = fs[stock_cols + market_cols].quantile([0.01, 0.25, 0.5, 0.75, 0.99]).T
q.columns = ["p1", "p25", "p50", "p75", "p99"]
q["표준편차"] = fs[stock_cols + market_cols].std()
q.round(4)

,p1,p25,p50,p75,p99,표준편차
ret_1,-0.0826,-0.0146,0.0000,0.0142,0.1038,0.0327
ret_5,-0.1639,-0.0327,0.0000,0.0363,0.2372,0.0720
ret_20,-0.2623,-0.0617,0.0019,0.0800,0.5558,0.1488
ret_60,-0.3507,-0.0829,0.0248,0.1667,1.1163,0.2908
ma_gap_5,-0.0861,-0.0159,0.0000,0.0164,0.1075,0.0346
ma_gap_20,-0.1634,-0.0367,-0.0003,0.0407,0.2709,0.0780
ma_gap_60,-0.2413,-0.0551,0.0052,0.0795,0.5261,0.1420
ret_20_skip5,-0.2305,-0.0543,0.0018,0.0684,0.4581,0.1259
vol_5,0.0042,0.0140,0.0219,0.0336,0.0966,0.0186
vol_20,0.0081,0.0184,0.0254,0.0350,0.0762,0.0143


## 5. 늦은 갱신 플래그

In [7]:
lu = fs.groupby(fs.trade_date.dt.to_period("M"))["late_update_20"].mean()
print(f"late_update_20 = True 비율: 전체 {fs.late_update_20.mean():.1%}")
lu[lu > 0].round(3).to_frame("월별 비율")

late_update_20 = True 비율: 전체 23.1%


,월별 비율
trade_date,
2024-04,0.985
2024-05,0.985
2024-06,0.985
2024-07,0.990
2024-08,0.896
2025-08,0.446
2025-09,0.495
2025-12,0.003
2026-01,0.005


## 6. 저장

In [8]:
feat.to_parquet(ROOT / "data" / "cache" / "features_train.parquet", index=False)
cand = spec.assign(**{"평균 IC": np.nan, "IC IR": np.nan, "안정성": "", "추천 등급": ""})
cand = cand[["변수명", "계열", "계산식", "필요 과거 기간", "평균 IC", "IC IR", "안정성", "추천 등급", "비고", "단위"]]
cand.to_csv(ROOT / "outputs" / "feature_candidates.csv", index=False, encoding="utf-8-sig")
cand.to_parquet(ROOT / "data" / "cache" / "feature_spec.parquet", index=False)   # Step 5가 읽는 원본 명세
print(feat.shape, "-> data/cache/features_train.parquet")
print(cand.shape, "-> outputs/feature_candidates.csv")

(121261, 234) -> data/cache/features_train.parquet
(133, 10) -> outputs/feature_candidates.csv


## 7. 요약과 한계 (v2)

- 계획서 Step 4의 계열을 모두 만들었다: 수익률·모멘텀, 변동성, 하방 특성, 가격 위치, 거래·유동성, 캔들, 시장·시장 대비, 수급(투자자별·기관 세부·프로그램), 공매도·대차, 신용, 업종. 종목 변수는 날짜별 백분위 순위 버전(`_rk`)도 만들었다.
- **v2에서 바뀐 점**
  - 가격을 KRX 정규장 일봉으로 바꿨고, 스파이크 보정을 하지 않는다.
  - 거래대금은 KRX 실제 값을 쓴다.
  - 수급 분모를 데이터 기준(통합 또는 KRX)에 맞췄다.
  - 시가총액과 회전율 변수를 추가했다(시점별 상장주식수).
  - 거래정지는 KRX의 `halted` 플래그로 판단한다.
- **만들지 않았거나 바꾼 것**
  - 120일 수익률, 52주 고점·저점: 60일 상한 결정에 따라 60일 버전으로 대체했다.
  - 대주(stockloan_*): 값이 거의 0이라 제외했다.
  - 재무 지표: DB에 없다.
- **가정**
  - 공매도 행이 없는 날은 공매도 0으로 채웠다.
  - 거래정지일에 프로그램 매매 행이 없으면 0으로 채웠다.
  - 업종은 현재값을 과거에도 쓴다.
- **주의**
  - 폐지·합병 3종목(003410, 010620, 042670)은 수급 데이터가 없어서 수급 변수가 비어 있다.
  - 공매도 변수는 2025-03-31 전후로 체제가 다르다.
  - 대차 변수는 2024년 1~7월 값이 사후 재갱신된 값이다(`late_update_20` 플래그).